# Stablecoin Stability Analysis - EDA

This notebook explores the historical price data of various stablecoins to identify trends, depeg events, and correlations.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import glob
import os

PROCESSED_DIR = '../data/processed'

def load_all_data():
    all_files = glob.glob(os.path.join(PROCESSED_DIR, '*.csv'))
    combined_df = pd.DataFrame()
    
    for filename in all_files:
        df = pd.read_csv(filename)
        base_name = os.path.basename(filename)
        symbol = base_name.replace('processed_', '').split('_')[0]
        df['symbol'] = symbol
        combined_df = pd.concat([combined_df, df], ignore_index=True)
        
    combined_df['timestamp'] = pd.to_datetime(combined_df['timestamp'])
    return combined_df

df = load_all_data()
df.head()

## 1. Price History
Visualizing the price movement of all stablecoins over the last 2 years.

In [ ]:
fig = px.line(df, x='timestamp', y='close', color='symbol', title='Stablecoin Price History')
fig.add_hline(y=1.0, line_dash="dash", line_color="red")
fig.show()

## 2. Depeg Events
Highlighting instances where the price deviated by more than 1%.

In [ ]:
depeg_df = df[df['deviation_pct'].abs() > 0.01]

fig = px.scatter(depeg_df, x='timestamp', y='close', color='symbol', title='Depeg Events (>1% Deviation)')
fig.add_hline(y=1.0, line_dash="dash", line_color="red")
fig.show()

## 3. Volatility Analysis
Examining the 24h rolling volatility.

In [ ]:
fig = px.line(df, x='timestamp', y='volatility_24h', color='symbol', title='24h Rolling Volatility')
fig.show()

## 4. Correlation Matrix
Analyzing how stablecoins move in relation to each other.

In [ ]:
pivot_df = df.pivot_table(index='timestamp', columns='symbol', values='close')
corr = pivot_df.corr()

fig = px.imshow(corr, text_auto=True, aspect="auto", title="Price Correlation Matrix")
fig.show()